In [1]:
import duckdb
import os
import pandas as pd

In [2]:
CURRENT_DIR = os.getcwd()
while not CURRENT_DIR.endswith("apple-retail-pipeline") and CURRENT_DIR != "/":
    CURRENT_DIR = os.path.dirname(CURRENT_DIR)
os.chdir(CURRENT_DIR)

WAREHOUSE_DIR = os.path.join(CURRENT_DIR, "data", "warehouse", "apple")

DB_PATH = os.path.join(WAREHOUSE_DIR, "apple_warehouse.duckdb")
con = duckdb.connect(DB_PATH)

In [3]:
tables = {
    "fact_sales": "fact_sales.parquet",
    "dim_store": "dim_store.parquet",
    "dim_product": "dim_product.parquet",
    "dim_category": "dim_category.parquet",
    "dim_warranty": "dim_warranty.parquet",
    "dim_date": "dim_date.parquet",
}

In [4]:
for name, fname in tables.items():
    path = os.path.join(WAREHOUSE_DIR, fname)
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM parquet_scan('{path}')")
print("All tables registered as views in DuckDB.\n")

All tables registered as views in DuckDB.



In [6]:
print("TOP 10 STORES WITH THE MOST REVENUE")
print(con.execute("""
SELECT s.store_name, SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_store s ON f.store_id = s.store_id
GROUP BY s.store_name
ORDER BY total_revenue DESC
LIMIT 10;
""").df(), "\n")

TOP 10 STORES WITH THE MOST REVENUE
             store_name  total_revenue
0       Apple Chadstone    165220023.0
1   Apple Covent Garden    165176160.0
2  Apple The Dubai Mall    163741158.0
3    Apple Orchard Road    162709375.0
4   Apple Central World    162552209.0
5  Apple Champs-Elysees    161720430.0
6      Apple Dubai Mall     84478026.0
7       Apple Southland     83830993.0
8           Apple Kyoto     83706593.0
9         Apple Fukuoka     83576408.0 

